# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² CRC Survivor dataset using the `mlcroissant` library. All references to entities (record sets, fields, etc.) use their `@id`, as recommended for clarity and reproducibility when working with Croissant datasets.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install -U mlcroissant

## 1. Data Loading

We first load dataset metadata and define the schema URL. The schema provides a FAIR, machine-readable description of available data splits and fields.

We use the `mlcroissant` library to programmatically explore and access records via their Croissant `@id`s.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'Dataset')}: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview

We will inspect the available record sets (i.e., Croissant's main entry points for tabular data) and explore their associated fields and `@id`s. All navigation is performed by referencing entities through their `@id` for full reproducibility.

Note: The specific record set `@id`s depend on the dataset schema. We'll enumerate available record sets and display their fields and columns' `@id`s.

In [ ]:
record_sets = list(dataset.record_sets)
if record_sets:
    print(f"Found {len(record_sets)} record set(s):")
    for rset in record_sets:
        print(f"  - @id: {rset['@id']}")
        print(f"    name: {rset.get('name','')}")
        print(f"    description: {rset.get('description','')}")
        fields = rset.get('field', [])
        # field can be a dict or list
        flds = fields if isinstance(fields, list) else [fields]
        print(f"    Fields/columns:")
        for f in flds:
            if isinstance(f, dict):
                print(f"      - @id: {f.get('@id','')} (name: {f.get('name','')})")
            else:
                print(f"      - @id: {f}")
else:
    print('No record sets defined in this dataset. Please check the schema.')

## 3. Data Extraction

Let's extract tabular data from the record set(s) using their Croissant `@id`. We'll load each record set into a `pandas.DataFrame`. If the dataset contains only one record set, we'll demonstrate extraction for that set.

> **Note:** Always use entity `@id`s for specifying record sets and fields; below, you'll see actual `@id`s in variables.

In [ ]:
# Get the list of record set @id's
record_set_ids = [rset['@id'] for rset in record_sets]
# For demonstration, we'll load all record sets
dfs = {}

for rset_id in record_set_ids:
    # Records are yielded as dicts; field names are the full Croissant field @id
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dfs[rset_id] = df
        print(f"Loaded {len(df)} records from record set: {rset_id}")
        print(f"Columns (`@id`): {df.columns.tolist()}\n")
    else:
        print(f"No records found for record set: {rset_id}\n")
# For further analysis, pick the first available record set
if dfs:
    primary_rset_id = record_set_ids[0]
    display(dfs[primary_rset_id].head())
else:
    primary_rset_id = None

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate data processing: filtering, normalizing a numeric field, and grouping by a categorical field. Replace the field `@id`s below with those found in the previous overview.

> **Tip:** When in doubt, check or print the `DataFrame`'s columns—they are the full field `@id`s from the schema.

In [ ]:
if primary_rset_id:
    df = dfs[primary_rset_id]
    print(f"Working with record set: {primary_rset_id}")
    print("Columns available:")
    print(df.columns.tolist())
    
    # For this demo, try to infer a numeric field (e.g., age, interval, etc.)
    import numpy as np
    numeric_col_id = None
    for col in df.columns:
        if df[col].dtype.kind in 'if' or pd.api.types.is_numeric_dtype(df[col]):
            numeric_col_id = col
            break
    if numeric_col_id is None:
        # Try to convert potential int/float columns by force
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_col_id = col
                    break
            except Exception:
                continue
    if numeric_col_id:
        print(f"Selected numeric field for EDA: {numeric_col_id}\n")
        # Pick a threshold based on distribution
        threshold = df[numeric_col_id].mean() if df[numeric_col_id].notna().sum() else 0
        filtered_df = df[df[numeric_col_id] > threshold]
        print(f"Filtered records with {numeric_col_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_col_id}_normalized"] = (
            (filtered_df[numeric_col_id] - filtered_df[numeric_col_id].mean()) / filtered_df[numeric_col_id].std()
        )
        print(f"Normalized {numeric_col_id} for filtered records:")
        display(filtered_df[[numeric_col_id, f"{numeric_col_id}_normalized"]].head())
        # Try to find a grouping categorical field (e.g. sex, type, etc.)
        cat_col_id = None
        for col in df.columns:
            if df[col].dtype == object and df[col].nunique() < 10:
                cat_col_id = col
                break
        if cat_col_id:
            print(f"Grouping by: {cat_col_id}")
            grouped = filtered_df.groupby(cat_col_id)[numeric_col_id].mean().to_frame("mean_"+numeric_col_id)
            display(grouped)
    else:
        print("No numeric field found for EDA demonstration.")
else:
    print("No record set available for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field if one was found, and the grouping if available.

In [ ]:
import matplotlib.pyplot as plt

if primary_rset_id and numeric_col_id:
    plt.figure(figsize=(7, 4))
    df[numeric_col_id].hist(bins=10, edgecolor='k')
    plt.title(f'Distribution of {numeric_col_id}')
    plt.xlabel(numeric_col_id)
    plt.ylabel('Count')
    plt.show()

    # If category column exists, do box plot
    if cat_col_id:
        plt.figure(figsize=(8, 5))
        df.boxplot(column=numeric_col_id, by=cat_col_id)
        plt.title(f'{numeric_col_id} by {cat_col_id}')
        plt.suptitle("")
        plt.xlabel(cat_col_id)
        plt.ylabel(numeric_col_id)
        plt.show()
else:
    print('No suitable numeric field found for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to:

- Access a FAIR Croissant dataset using only `@id`-based references for all entities.
- Load and explore record set structure, field names, and data values.
- Filter and normalize a numeric field and group data by a categorical (where possible).
- Visualize common relationships in the clinical dataset.

This approach ensures full reproducibility and clarity in FAIR data workflows. For deeper analysis, select specific `@id`s and further process clinical and molecular variables as needed.